# Day 3 — Probability for Machine Learning
**30-Day ML & AI Mastery Plan · Shubham**

Every ML classifier outputs a probability, and every model is trained by asking "which parameters make the observed data most probable?" Today you learn the language all of that is written in, and you prove every formula by **simulation** instead of taking it on trust.

| Part | What you'll learn | Time |
|---|---|---|
| 1 | Experiments, events, the 3 rules of probability | 15 min |
| 2 | Mutually exclusive vs non-exclusive events | 10 min |
| 3 | Independent vs dependent events, conditional probability | 15 min |
| 4 | **Bayes' theorem** and the base-rate trap | 20 min |
| 5 | Random variables: discrete vs continuous | 10 min |
| 6 | PMF, PDF and CDF | 20 min |
| 7 | The 4 distributions you'll meet constantly | 25 min |
| **8** | **Live demo: a Naive Bayes spam filter, by hand** | 40 min |
| 9 | Exercises | 30–60 min |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from collections import Counter
import math, re

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(seed=3)

---
# Part 1 — Experiments, outcomes, events
- **Experiment**: anything with an uncertain result (roll a die, a customer visits, an email arrives).
- **Sample space S**: every possible outcome. For one die, S = {1, 2, 3, 4, 5, 6}.
- **Event**: a set of outcomes you care about, e.g. "even" = {2, 4, 6}.
- **Probability** P(A) = (outcomes in A) / (all outcomes), when all outcomes are equally likely.

**The 3 rules (axioms)**: 0 ≤ P(A) ≤ 1 · P(S) = 1 · for events that can't co-occur, probabilities add.
Useful consequence, the **complement rule**: P(not A) = 1 − P(A).

In [ ]:
S = {1, 2, 3, 4, 5, 6}
even = {2, 4, 6}
print("P(even) by counting:", len(even) / len(S))

# Frequentist view: probability = long-run frequency. Simulate 100,000 rolls.
rolls = rng.integers(1, 7, size=100_000)
print("P(even) by simulation:", np.isin(rolls, list(even)).mean().round(4))

In [ ]:
# Law of large numbers: the frequency settles down to the true probability as n grows
running = np.cumsum(rolls % 2 == 0) / np.arange(1, len(rolls) + 1)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(running[:5000], lw=1)
ax.axhline(0.5, color="red", ls="--", label="true P = 0.5")
ax.set(title="Law of large numbers: running fraction of even rolls", xlabel="number of rolls", ylabel="fraction even")
ax.legend(); plt.show()

In [ ]:
# Complement rule in action: P(at least one six in 4 rolls) = 1 - P(no six in 4 rolls)
exact = 1 - (5/6) ** 4
sim = (rng.integers(1, 7, size=(100_000, 4)) == 6).any(axis=1).mean()
print(f"exact {exact:.4f} | simulated {sim:.4f}")

---
# Part 2 — Mutually exclusive vs non-exclusive events
**Mutually exclusive**: can't both happen (a roll can't be both 1 and 6). Then **P(A or B) = P(A) + P(B)**.

**Non-exclusive**: can overlap. Adding double-counts the overlap, so use the **addition rule**:

**P(A or B) = P(A) + P(B) − P(A and B)**

In [ ]:
A = {2, 4, 6}          # even
B = {4, 5, 6}          # greater than 3
P = lambda E: len(E) / 6

print("overlap A∩B:", A & B)
print("naive P(A)+P(B):           ", P(A) + P(B))              # 1.0: wrong, counts 4 and 6 twice
print("addition rule:             ", P(A) + P(B) - P(A & B))   # 0.667
print("direct count of A∪B:       ", P(A | B))

one, six = {1}, {6}
print("\nmutually exclusive (1 or 6):", P(one) + P(six), "=", P(one | six))

---
# Part 3 — Independent vs dependent events, conditional probability
**Conditional probability**: the probability of A *given that* B happened. You shrink the world to B:

**P(A | B) = P(A and B) / P(B)**

**Independent** events don't inform each other: P(A | B) = P(A), and then **P(A and B) = P(A) · P(B)**.
**Dependent** events do: the multiplication must use the conditional, **P(A and B) = P(A) · P(B | A)**.

Classic example: drawing two cards **with** replacement (independent) vs **without** replacement (dependent).

In [ ]:
# Two aces in a row from a 52-card deck
with_repl = (4/52) * (4/52)
without = (4/52) * (3/51)             # after one ace is gone, only 3 aces in 51 cards
print(f"with replacement (independent): {with_repl:.5f}")
print(f"without replacement (dependent): {without:.5f}")

# Verify 'without replacement' by simulation
deck = np.array([1]*4 + [0]*48)        # 1 = ace
trials = 500_000
first_two = rng.random((trials, 52)).argsort(axis=1)[:, :2]   # a random shuffle per trial, keep 2 cards
print(f"simulated ({trials:,} trials):     {(deck[first_two].sum(axis=1) == 2).mean():.5f}")

In [ ]:
# Conditional probability on REAL data: Titanic
titanic = sns.load_dataset("titanic")
p_surv = titanic.survived.mean()
p_surv_given_female = titanic.loc[titanic.sex == "female", "survived"].mean()
p_surv_given_male = titanic.loc[titanic.sex == "male", "survived"].mean()
print(f"P(survived)          = {p_surv:.3f}")
print(f"P(survived | female) = {p_surv_given_female:.3f}")
print(f"P(survived | male)   = {p_surv_given_male:.3f}")
print("Independent?", np.isclose(p_surv, p_surv_given_female, atol=0.02), "-> knowing sex changes the probability, so they are DEPENDENT")

A feature is useful to a model exactly when the target is **dependent** on it. If P(survived | feature) were the same for every value of the feature, the feature would carry no information. This is the idea behind feature selection on Day 10.

---
# Part 4 — Bayes' theorem
Bayes turns a probability around: from P(evidence | cause), which is easy to measure, to P(cause | evidence), which is what you actually want.

**P(A | B) = P(B | A) · P(A) / P(B)**

| Term | Name | Meaning |
|---|---|---|
| P(A) | prior | belief before the evidence |
| P(B \| A) | likelihood | how expected the evidence is if A is true |
| P(B) | evidence | how common the evidence is overall |
| P(A \| B) | posterior | updated belief after the evidence |

P(B) is usually found with the **law of total probability**: P(B) = P(B | A)·P(A) + P(B | not A)·P(not A).

### The base-rate trap: a medical test
A disease affects **1%** of people. A test catches **99%** of real cases (sensitivity) and wrongly flags **5%** of healthy people (false positive rate). You test positive. How likely is it that you have the disease? Guess before running the cell.

In [ ]:
def posterior(prior, sensitivity, false_pos_rate):
    p_pos = sensitivity * prior + false_pos_rate * (1 - prior)   # law of total probability
    return sensitivity * prior / p_pos

print(f"P(disease | positive) = {posterior(0.01, 0.99, 0.05):.1%}")

In [ ]:
# Why so low? Count 10,000 people instead of using formulas ("natural frequencies")
N = 10_000
sick = int(N * 0.01); healthy = N - sick
true_pos = round(sick * 0.99); false_pos = round(healthy * 0.05)
print(f"{sick} sick -> {true_pos} test positive")
print(f"{healthy} healthy -> {false_pos} test positive (false alarms)")
print(f"Of {true_pos + false_pos} positives, only {true_pos} are sick = {true_pos/(true_pos+false_pos):.1%}")

In [ ]:
# The prior drives the answer: same test, different base rates
priors = np.array([0.001, 0.01, 0.05, 0.1, 0.3, 0.5])
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(priors * 100, [posterior(p, 0.99, 0.05) * 100 for p in priors], marker="o")
ax.set(title="Same 99%-sensitive test: posterior depends on the base rate",
       xlabel="prior: % of population with the disease", ylabel="P(disease | positive) %", xscale="log")
for p in priors:
    ax.annotate(f"{posterior(p,0.99,0.05):.0%}", (p*100, posterior(p,0.99,0.05)*100), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=9)
plt.show()

**ML link:** this is exactly the problem of **imbalanced classes**. A fraud model with 99% recall still produces mostly false alarms when only 0.1% of transactions are fraud. That's why on Day 9 you'll judge such models by **precision** and **recall**, not accuracy.

---
# Part 5 — Random variables
A **random variable** turns outcomes into numbers so we can do maths on them.

| | Discrete | Continuous |
|---|---|---|
| Values | countable: 0, 1, 2, … | any value in a range |
| Examples | number of defects, clicks, spam words | height, time, temperature, price |
| Described by | **PMF**: P(X = x) | **PDF**: density f(x) |
| P(X = exactly x) | can be > 0 | always 0 |
| Probability of a range | sum of PMF values | area under the PDF |

Both have a **CDF**: F(x) = P(X ≤ x), which always rises from 0 to 1.

---
# Part 6 — PMF, PDF and CDF
**PMF** (discrete): bar heights are probabilities and they sum to 1.
**PDF** (continuous): curve height is a *density*, not a probability. It can exceed 1. Only **areas** are probabilities.
**CDF**: the running total. P(a < X ≤ b) = F(b) − F(a).

In [ ]:
# Discrete: the sum of two dice. Build the PMF by counting all 36 outcomes.
sums = [a + b for a in range(1, 7) for b in range(1, 7)]
pmf = pd.Series(Counter(sums)).sort_index() / 36
cdf = pmf.cumsum()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].bar(pmf.index, pmf.values, color="steelblue")
axes[0].set(title="PMF: sum of two dice", xlabel="sum", ylabel="P(X = x)")
axes[1].step(cdf.index, cdf.values, where="post", color="darkorange")
axes[1].set(title="CDF: P(X ≤ x)", xlabel="sum", ylabel="cumulative probability")
plt.tight_layout(); plt.show()
print("PMF sums to", pmf.sum(), "| P(sum = 7) =", round(pmf[7], 4), "| P(sum ≤ 4) =", round(cdf[4], 4))

In [ ]:
# Continuous: a narrow normal distribution shows that density can exceed 1
narrow = stats.norm(loc=0, scale=0.2)
print("density at 0:", round(narrow.pdf(0), 3), "<- more than 1, and that's fine")
print("total area:  ", round(stats.norm(0, 0.2).cdf(10) - stats.norm(0, 0.2).cdf(-10), 6))

# Probability of a range = area under the PDF = difference of CDFs
marks = stats.norm(loc=70, scale=10)
x = np.linspace(30, 110, 400)
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(x, marks.pdf(x), color="navy")
band = (x >= 60) & (x <= 80)
ax.fill_between(x[band], marks.pdf(x[band]), alpha=0.3, color="navy")
p = marks.cdf(80) - marks.cdf(60)
ax.set(title=f"Marks ~ N(70, 10): P(60 ≤ X ≤ 80) = F(80) − F(60) = {p:.3f}", xlabel="mark", ylabel="density")
plt.show()

---
# Part 7 — Four distributions you'll meet constantly
| Distribution | Type | Models | Parameters | Mean | ML example |
|---|---|---|---|---|---|
| **Bernoulli** | discrete | one yes/no trial | p | p | one email: spam or not; logistic regression output |
| **Binomial** | discrete | number of successes in n trials | n, p | n·p | conversions out of 1,000 visitors |
| **Poisson** | discrete | count of events in a fixed interval | λ | λ | support tickets per hour, defects per batch |
| **Normal** | continuous | sums of many small effects | μ, σ | μ | measurement error, weight initialisation, linear-regression noise |

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.4))

axes[0].bar([0, 1], stats.bernoulli(0.3).pmf([0, 1]), color=["grey", "crimson"])
axes[0].set(title="Bernoulli(p=0.3)", xticks=[0, 1], xticklabels=["not spam", "spam"])

k = np.arange(0, 21)
axes[1].bar(k, stats.binom(20, 0.3).pmf(k), color="steelblue")
axes[1].set(title="Binomial(n=20, p=0.3)", xlabel="successes")

k = np.arange(0, 16)
for lam, c in [(2, "teal"), (5, "orange")]:
    axes[2].plot(k, stats.poisson(lam).pmf(k), "o-", color=c, label=f"λ={lam}")
axes[2].set(title="Poisson", xlabel="events per interval"); axes[2].legend()

x = np.linspace(-6, 6, 300)
for mu, sd, c in [(0, 1, "navy"), (0, 2, "purple"), (2, 0.7, "green")]:
    axes[3].plot(x, stats.norm(mu, sd).pdf(x), color=c, label=f"μ={mu}, σ={sd}")
axes[3].set(title="Normal"); axes[3].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Answering real questions with scipy.stats: .pmf / .pdf / .cdf / .sf (= 1 - cdf) / .ppf (inverse cdf)

# Binomial: a page converts 3% of visitors. Of 200 visitors today, what's P(at least 10 convert)?
print("P(>=10 conversions):", round(stats.binom(200, 0.03).sf(9), 4))

# Poisson: a server averages 4 errors/hour. P(more than 8 in an hour)? That would trigger an alert.
print("P(>8 errors):       ", round(stats.poisson(4).sf(8), 4))

# Normal: delivery times ~ N(48h, 6h). What time covers 95% of deliveries? (inverse CDF)
print("95th percentile:    ", round(stats.norm(48, 6).ppf(0.95), 1), "hours")

# The 68-95-99.7 rule
for k in (1, 2, 3):
    print(f"P(within {k}σ) = {stats.norm.cdf(k) - stats.norm.cdf(-k):.4f}")

In [ ]:
# Simulation check: a binomial is just a sum of Bernoullis
n, p = 20, 0.3
sim = rng.binomial(1, p, size=(100_000, n)).sum(axis=1)
emp = pd.Series(sim).value_counts(normalize=True).sort_index()
k = np.arange(0, 15)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(emp.index, emp.values, alpha=0.5, label="simulated: sum of 20 Bernoullis")
ax.plot(k, stats.binom(n, p).pmf(k), "ro-", label="Binomial(20, 0.3) PMF")
ax.set(title="Binomial = sum of independent Bernoulli trials", xlabel="successes"); ax.legend(); plt.show()

---
# Part 8 — LIVE DEMO: a Naive Bayes spam filter, by hand
This is how early email spam filters worked, and Naive Bayes is still a strong baseline for text classification.

**The idea:** for an email with words w₁ … wₙ, compare
P(spam | words) ∝ P(spam) · P(w₁ | spam) · P(w₂ | spam) · …
P(ham | words) ∝ P(ham) · P(w₁ | ham) · P(w₂ | ham) · …
and pick the larger one.

**"Naive"** = we assume words are independent of each other given the class. That's false ("free" and "prize" go together), but it works surprisingly well, and it makes the maths tractable.

Three practical fixes you'll implement:
1. **Laplace smoothing**: add 1 to every word count, so an unseen word doesn't make a probability exactly 0 and wipe out the whole product.
2. **Log probabilities**: multiplying many small numbers underflows to 0.0; adding their logs doesn't.
3. **Normalising** the two scores back into a probability that sums to 1.

### Step 1 — Training data

In [ ]:
train = [
    ("Win a FREE prize now, click here to claim your reward", "spam"),
    ("Congratulations you won a lottery cash prize claim now", "spam"),
    ("Limited offer buy cheap meds online free shipping", "spam"),
    ("Earn money fast from home click the link now", "spam"),
    ("URGENT your account is suspended verify your password now", "spam"),
    ("Exclusive deal free gift card claim before midnight", "spam"),
    ("Get rich quick guaranteed cash bonus click here", "spam"),
    ("Hi team, the sprint review meeting is moved to Friday", "ham"),
    ("Can you share the Q3 report before the meeting", "ham"),
    ("Lunch tomorrow? Let me know what time works", "ham"),
    ("Please review my pull request for the API changes", "ham"),
    ("The client approved the project plan, thanks everyone", "ham"),
    ("Reminder: submit your timesheet by Friday evening", "ham"),
    ("Here are the notes from today's design discussion", "ham"),
    ("Your order has shipped and will arrive on Monday", "ham"),
    ("Can we move our call to 4 pm? I have a clash", "ham"),
]
df = pd.DataFrame(train, columns=["text", "label"])
df["label"].value_counts()

### Step 2 — Tokenise and count words per class

In [ ]:
STOPWORDS = {"a", "the", "to", "your", "you", "is", "for", "and", "of", "on", "by", "my", "me", "i", "we", "our", "from", "can", "be", "will", "has", "are", "here", "now"}

def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [w for w in words if w not in STOPWORDS]

word_counts = {"spam": Counter(), "ham": Counter()}
for text, label in train:
    word_counts[label].update(tokenize(text))

vocab = set(word_counts["spam"]) | set(word_counts["ham"])
print("vocabulary size:", len(vocab))
print("top spam words:", word_counts["spam"].most_common(6))
print("top ham words: ", word_counts["ham"].most_common(6))

### Step 3 — Train: priors and smoothed likelihoods

In [ ]:
class NaiveBayesSpam:
    def fit(self, texts, labels, alpha=1.0):
        self.classes = sorted(set(labels))
        self.alpha = alpha
        self.counts = {c: Counter() for c in self.classes}
        for t, c in zip(texts, labels):
            self.counts[c].update(tokenize(t))
        self.vocab = set().union(*self.counts.values())
        n = len(labels)
        self.log_prior = {c: math.log(labels.count(c) / n) for c in self.classes}
        self.total = {c: sum(self.counts[c].values()) for c in self.classes}
        return self

    def log_likelihood(self, word, c):
        # Laplace smoothing: (count + alpha) / (total words in class + alpha * vocab size)
        return math.log((self.counts[c][word] + self.alpha) / (self.total[c] + self.alpha * len(self.vocab)))

    def predict_proba(self, text):
        scores = {c: self.log_prior[c] + sum(self.log_likelihood(w, c) for w in tokenize(text))
                  for c in self.classes}
        m = max(scores.values())                              # log-sum-exp trick for stability
        exp = {c: math.exp(s - m) for c, s in scores.items()}
        z = sum(exp.values())
        return {c: v / z for c, v in exp.items()}

    def predict(self, text):
        proba = self.predict_proba(text)
        return max(proba, key=proba.get)

    def explain(self, text, top=5):
        rows = [(w, self.log_likelihood(w, "spam") - self.log_likelihood(w, "ham")) for w in set(tokenize(text))]
        return sorted(rows, key=lambda r: abs(r[1]), reverse=True)[:top]

texts, labels = list(df.text), list(df.label)
nb = NaiveBayesSpam().fit(texts, labels)
print("priors:", {c: round(math.exp(v), 3) for c, v in nb.log_prior.items()})

### Step 4 — Classify new emails

In [ ]:
tests = [
    "Claim your free cash prize now",
    "Can you review the project report before Friday",
    "Free lunch at the team meeting tomorrow",
    "Verify your password to claim a bonus",
    "Quarterly roadmap discussion moved to Monday",
]
for t in tests:
    p = nb.predict_proba(t)
    print(f"{nb.predict(t):>4}  P(spam)={p['spam']:.3f}  | {t}")

### Step 5 — Explain a decision
Each word's **log-likelihood ratio** log P(w|spam) − log P(w|ham) shows how hard it pushes toward spam (positive) or ham (negative). This interpretability is a big reason Naive Bayes is still used.

In [ ]:
for word, llr in nb.explain("Free lunch at the team meeting tomorrow"):
    direction = "→ spam" if llr > 0 else "→ ham"
    print(f"{word:>10}: {llr:+.2f} {direction}")

### Step 6 — Why the log trick matters

In [ ]:
tiny = 1e-5
print("product of 100 small probabilities:", tiny ** 100, " <- underflows to 0.0")
print("sum of their logs:                 ", round(100 * math.log(tiny), 2), " <- perfectly fine")

### Step 7 — Check against scikit-learn
scikit-learn's `MultinomialNB` implements the same maths. If our hand-built version agrees, we understood it correctly.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vec = CountVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
X = vec.fit_transform(texts)
sk = MultinomialNB(alpha=1.0).fit(X, labels)
sk_proba = sk.predict_proba(vec.transform(tests))[:, list(sk.classes_).index("spam")]

for t, ours_p, sk_p in zip(tests, [nb.predict_proba(t)["spam"] for t in tests], sk_proba):
    print(f"ours {ours_p:.4f} | sklearn {sk_p:.4f} | {t}")

The two tiny differences come from one design choice: scikit-learn **ignores words it never saw in training** ("at", "quarterly", "roadmap"), while our version gives each unseen word a smoothed probability. Both are valid; ignoring unseen words is the more common convention.

**What you just built uses every idea from today:** priors and Bayes' theorem (Part 4), conditional probability P(word | class) (Part 3), the independence assumption (Part 3), a Bernoulli outcome (Part 7), and the complement rule in the normalisation step (Part 1).

---
# Part 9 — Exercises
Paste your code in chat for review.

**Ex 1 — Addition rule.** In a class of 60, 25 study ML, 20 study DL and 8 study both. What's P(a random student studies ML or DL)? Compute it with the formula, then verify by building the sets and counting.

**Ex 2 — Bayes.** 2% of transactions are fraud. A model flags 95% of frauds and 3% of genuine transactions. If a transaction is flagged, what's the probability it's fraud? Then: what false-positive rate would the model need for a flagged transaction to be fraud at least 50% of the time?

**Ex 3 — Distributions.** A call centre gets 12 calls per hour on average. Using `scipy.stats`: (a) P(exactly 15 calls in an hour)? (b) P(fewer than 8)? (c) Simulate 10,000 hours with NumPy and check (b).

**Ex 4 — Spam filter.** Add 4 new emails of your own (2 spam, 2 ham) to the training data, retrain, and find one test email whose prediction **changes**. Use `explain()` to show which word caused the flip.

**Stretch.** The Monty Hall problem: you pick 1 of 3 doors; the host opens a different door with a goat; you can switch. Simulate 100,000 games and estimate P(win) for "stay" vs "switch". Then explain the result with conditional probability.

In [ ]:
# Ex 1

In [ ]:
# Ex 2

In [ ]:
# Ex 3

In [ ]:
# Ex 4

In [ ]:
# Stretch: Monty Hall

---
### Day 3 recap
- **Probability** = long-run frequency; simulate to check any formula.
- **Complement:** P(at least one) = 1 − P(none).
- **Addition rule:** P(A or B) = P(A) + P(B) − P(A and B). Drop the last term only if they're mutually exclusive.
- **Conditional:** P(A | B) = P(A and B) / P(B). Independent means P(A | B) = P(A).
- **Bayes:** posterior ∝ likelihood × prior. A rare event stays fairly unlikely even after strong evidence (the base-rate trap).
- **PMF** (discrete) gives probabilities; **PDF** (continuous) gives densities, and only areas are probabilities; **CDF** = P(X ≤ x).
- **Bernoulli, Binomial, Poisson, Normal**: know what each one models.
- **Naive Bayes** = Bayes + an independence assumption + Laplace smoothing + logs.
- **Tomorrow (Day 4): Statistics.** Centre and spread, normalisation vs standardisation, the Central Limit Theorem, confidence intervals and hypothesis tests.